In [ ]:
# ===== BASELINE GREEDY — CONFIG (edit ONLY this cell) =====================

REPO_URL = "https://github.com/Avi161/ACSolverX.git"
BRANCH   = "test/stable-ac-moves-w4"   # branch that contains experiments/
REPO_DIR = "ACSolverX"
CLONE       = True
UPDATE_REPO = True           # git reset --hard so a RESTART pulls latest push
MOUNT_DRIVE = True           # results -> Drive if True, else local ./results/

# --- experiment knobs ---------------------------------------------------
BUDGET = [1000, 5000, 10000, 25000, 50000]   # node budget per presentation; one jsonl per budget

cfg = {
    "DATASET": "data/ms640_solved.txt",
    "SUBSET": None,                 # None=all, list[int], or (start, end)
    "MAX_RELATOR_LENGTH": 24,       # per-relator cap (24 = ms640 layout)
    "CYCLIC_REDUCE": True,          # toggle cyclic reduction after each move

    # jsonl field toggles (each keeps BOTH the length + the relator pair)
    "use_min_relator": True,            # keeps min_relator_length + min_relator
    "use_max_relator": True,            # keeps max_relator_length + max_relator
    "use_max_relator_expanded": True,   # keeps max_relator_length_expanded + max_relator_expanded (longest state popped/expanded)
    "use_time": True,
    "use_path": True,
    "PATH_IN_SEPARATE_FILE": True,  # solved paths -> *_paths.jsonl

    "RESUME": True,                 # skip pres_ids already in the jsonl

    # --- HEAVY RUNS ONLY (e.g. the 261 unsolved reps at BUDGET=[1_000_000]) ---
    # True  -> memory-lean + parallel solver. Same solved / nodes_explored /
    #          min+max relator lengths as the normal path (verified on 400 runs);
    #          ~2.6x less RAM, ~2.4x faster. Paths are recovered afterwards by
    #          re-solving any presentation that actually solved.
    # False -> the normal pipeline, unchanged. Use this for the <=50k budgets.
    "HIGH_SPEEDUP": False,
    "N_WORKERS": 0,                 # 0 = auto (capped by RAM / GB_PER_PRES)
    "GB_PER_PRES": 9.0,             # measured @ 1M nodes, mrl=48, heavy solver
    "MP_START_METHOD": None,        # None = OS default (fork). Use "forkserver"
                                   # if workers hang right after starting.

    "PROGRESS_EVERY": 10,           # print a status line every N presentations

    # Live nodes/s WHILE a presentation is still running. 0 = off.
    # PROGRESS_EVERY only prints once a presentation finishes, which at a 1M
    # budget is minutes of silence -- set this (e.g. 30 or 60) on heavy runs to
    # watch throughput and get a per-presentation ETA. Costs nothing measurable.
    "HEARTBEAT_EVERY_S": 0,

    # Diagnose a silent heartbeat: prints a [hb-dbg] line every 5s (messages
    # received, queue depth, live presentations), prints a line when each
    # worker picks up a presentation, and emits the first sample on the very
    # first 1024-node tick instead of waiting ~10s. Leave False for real runs.
    "HEARTBEAT_DEBUG": False,

    # output
    "MOUNT_DRIVE": MOUNT_DRIVE,
    "DRIVE_OUT_DIR": "/content/drive/MyDrive/acsolverx_results/greedy_baseline",
    "LOCAL_OUT_DIR": "results/greedy_baseline",

    # Weights & Biases
    "USE_WANDB": True,
    "AUTO_AUTHENTICATE_WANDB": False,  # True: promptless via Colab Secret WANDB_API_KEY.
                                     # False: prompt for the API key on EVERY run of the SETUP cell.
    "WANDB_ENTITY": "avigyapaudel045-aisc",   # writable team entity (org-managed acct); None = account default
    "WANDB_PROJECT": "acsolver",
    "WANDB_MODE": "online",         # "offline" for flaky Colab -> wandb sync later
    "WANDB_GROUP": None,            # default: greedy_baseline_{date}
}

# --- to run the 261 hard presentations at a 1M budget, swap in: ------------
# BUDGET = [1_000_000]
# cfg["DATASET"]            = "data/ms_unsolved_reps/ms_reps_unsolved.txt"
# cfg["SUBSET"]             = None      # all 261
# cfg["MAX_RELATOR_LENGTH"] = 48
# cfg["HIGH_SPEEDUP"]       = True      # required: normal mode needs ~25 GB/presentation
# cfg["PROGRESS_EVERY"]     = 1
# cfg["HEARTBEAT_EVERY_S"]  = 60        # live nodes/s + ETA during each solve
# cfg["HEARTBEAT_DEBUG"]    = False     # True only to debug a silent heartbeat
# Use a High-RAM runtime. Pilot with cfg["SUBSET"] = (0, 2) first.

print("config loaded.")

In [7]:
# ==================== SETUP (clone / pull / install / mount) ==============
import os, sys, subprocess

def sh(cmd):
    print("$", cmd)
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if p.stdout: print(p.stdout[-2000:])
    if p.returncode != 0 and p.stderr: print("STDERR:", p.stderr[-2000:])

try:
    import google.colab  # noqa
    IN_COLAB = True
except Exception:
    IN_COLAB = False
print("Colab:", IN_COLAB)

if IN_COLAB:
    BASE = "/content"
    os.chdir(BASE)                       # anchor so re-runs never nest the clone
    if not os.path.isdir(REPO_DIR):
        if CLONE:
            sh(f"git clone --branch {BRANCH} --depth 1 {REPO_URL} {REPO_DIR}")
    elif UPDATE_REPO:
        sh(f"cd {REPO_DIR} && git fetch --depth 1 origin {BRANCH} && git reset --hard FETCH_HEAD")
    sh(f"cd {REPO_DIR} && git log -1 --oneline")
    sh("pip -q install numba numpy wandb")
    if MOUNT_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
    REPO_ROOT = os.path.join(BASE, REPO_DIR)
else:
    # local: walk up from cwd to the repo root (dir holding experiments/ + data/)
    REPO_ROOT = os.getcwd()
    while REPO_ROOT != "/" and not (
        os.path.isdir(os.path.join(REPO_ROOT, "experiments"))
        and os.path.isdir(os.path.join(REPO_ROOT, "data"))
    ):
        REPO_ROOT = os.path.dirname(REPO_ROOT)

# run from repo root so "data/..." and "import experiments..." resolve
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
print("repo root:", REPO_ROOT)

# SETUP's `git reset --hard` rewrites the .py files on disk, but Python keeps the
# OLD module objects in sys.modules for the life of the runtime -- so the RUN
# cell's `from experiments.run_baseline import run_dataset` would silently reuse
# stale code (a pull is NOT a reload). Drop them so the next import reads what
# SETUP just fetched. Without this you must Runtime -> Restart session.
import importlib
for _m in [m for m in sys.modules if m == "experiments" or m.startswith("experiments.")]:
    del sys.modules[_m]
importlib.invalidate_caches()

# ---- W&B authentication -------------------------------------------------
# AUTO_AUTHENTICATE_WANDB (set in CONFIG):
#   True  -> PROMPTLESS auth from a Colab Secret named WANDB_API_KEY (persists
#            across runtime restarts). Create it ONCE: Colab left sidebar ->
#            key icon (Secrets) -> add WANDB_API_KEY = <your key> -> toggle
#            'Notebook access' ON. Without a secret it reuses a valid key from
#            this session, else prompts once (a stale/invalid key is discarded,
#            not reused -- so it can always recover).
#   False -> prompt for the API key EVERY run (use to switch/refresh a key).
if cfg["USE_WANDB"]:
    import re, wandb

    def _clean_key(k):
        # strip whitespace/newlines and stray surrounding quotes from a paste
        return (k or "").strip().strip('"').strip("'").strip()

    def _valid(k):
        return bool(re.fullmatch(r"[A-Za-z0-9_]+", k or ""))

    def _prompt_key():
        import getpass
        return _clean_key(getpass.getpass(
            "Paste your W&B API key (from wandb.ai/authorize), then Enter: "))

    _auto = cfg.get("AUTO_AUTHENTICATE_WANDB", True)
    if not _auto:
        os.environ["WANDB_API_KEY"] = _prompt_key()          # always ask for a fresh key
    else:
        # auto: prefer a Colab Secret; else reuse a VALID session key; else prompt.
        _key, _why = None, ""
        try:
            from google.colab import userdata
            _key = _clean_key(userdata.get("WANDB_API_KEY"))
        except Exception as _e:
            _why = type(_e).__name__   # SecretNotFoundError / NotebookAccessError
        if _key:
            os.environ["WANDB_API_KEY"] = _key
            print("W&B: using Colab Secret WANDB_API_KEY (promptless).")
        elif _valid(os.environ.get("WANDB_API_KEY", "")):
            print("W&B: reusing the key from this session.")
        else:
            if os.environ.get("WANDB_API_KEY"):     # stale/invalid -> drop it, don't reuse
                print("W&B: discarding an invalid key left in this session.")
                os.environ.pop("WANDB_API_KEY", None)
            print("W&B: no usable Colab Secret WANDB_API_KEY"
                  f"{(' (' + _why + ')') if _why else ''}.")
            print("     -> Promptless auth: Colab left sidebar -> key icon (Secrets)")
            print("        -> Add  name=WANDB_API_KEY  value=<key from wandb.ai/authorize>")
            print("        -> toggle 'Notebook access' ON, then re-run this cell.")
            print("     Pasting once for THIS session instead:")
            os.environ["WANDB_API_KEY"] = _prompt_key()

    # final format check before hitting the server (Colab getpass can mangle a paste)
    if not _valid(os.environ.get("WANDB_API_KEY", "")):
        print("W&B: that key still has invalid characters (allowed: A-Z a-z 0-9 _).")
        print("     Re-copy it EXACTLY from https://wandb.ai/authorize.")

    # verify; relogin=True overwrites any stale key cached in ~/.netrc
    try:
        wandb.login(key=os.environ.get("WANDB_API_KEY", ""), relogin=True, verify=True)
        try:
            _default = wandb.Api().default_entity
        except Exception:
            _default = None
        _target = cfg["WANDB_ENTITY"] or _default
        print(f"W&B: authenticated ✓  runs -> {_target}/{cfg['WANDB_PROJECT']}")
    except Exception as e:
        print(f"W&B: authentication FAILED ✗ -- {e}")
        print("     tip: add a Colab Secret WANDB_API_KEY (promptless), or set "
              "AUTO_AUTHENTICATE_WANDB=False and re-run to paste a fresh key.")

Colab: True
$ cd ACSolverX && git fetch --depth 1 origin test/stable-ac-moves-w4 && git reset --hard FETCH_HEAD
HEAD is now at a0c8a6d W&B auth self-heals: prefer Colab Secret, discard stale/invalid env key

$ cd ACSolverX && git log -1 --oneline
a0c8a6d W&B auth self-heals: prefer Colab Secret, discard stale/invalid env key

$ pip -q install numba numpy wandb
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
repo root: /content/ACSolverX


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: avigyapaudel045 (avigyapaudel045-aisc) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B: authenticated ✓  runs -> avigyapaudel045-aisc/acsolver


In [ ]:
# ==================== RUN =================================================
from experiments.run_baseline import run_dataset

for budget in BUDGET:
    run_dataset(cfg, node_budget=budget)   # resumable; one jsonl per budget